In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import tifffile
import cv2
import os
from tqdm import tqdm
import argparse
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

In [ ]:
image_folder = './data/SAM_masks_1000'
output_directory = './data/results_16dVAE_cleaned'
quality_control_path = './data/mean_T1_originalfile.csv'
model_path = './data/results_16dVAE_cleaned/cvae_16d_best.weights.h5'
#model_path = "/content/cvae_16d_best.weights.h5"
batch_size = 16
latent_dim = 16
num_samples = 5

### VAE

In [ ]:
class CVAE(tf.keras.Model):
    """Convoutional Variational Autoencoder with input shape (256, 256, 1)."""

    def __init__(self, latent_dim):
        super(CVAE, self).__init__()

        self.latent_dim = latent_dim
        self.encoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(256, 256, 1)),
            tf.keras.layers.Conv2D(32, 3, strides=2, activation='relu'),
            tf.keras.layers.Conv2D(64, 3, strides=2, activation='relu'),
            tf.keras.layers.Conv2D(128, 3, strides=2, activation='relu'),
            tf.keras.layers.Conv2D(256, 3, strides=2, activation='relu'),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(latent_dim + latent_dim),
        ])

        self.decoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(latent_dim,)),
            tf.keras.layers.Dense(16 * 16 * 256, activation='relu'),
            tf.keras.layers.Reshape((16, 16, 256)),
            tf.keras.layers.Conv2DTranspose(256, 4, strides=2, padding='same', activation='relu'),
            tf.keras.layers.Conv2DTranspose(128, 4, strides=2, padding='same', activation='relu'),
            tf.keras.layers.Conv2DTranspose(64, 4, strides=2, padding='same', activation='relu'),
            tf.keras.layers.Conv2DTranspose(1, 4, strides=2, padding='same'),
        ])

    @tf.function
    def sample(self, eps=None):
        if eps is None:
            eps = tf.random.normal(shape=(100, self.latent_dim))
        return self.decode(eps, apply_sigmoid=True)

    def encode(self, x):
        mean, logvar = tf.split(self.encoder(x), num_or_size_splits=2, axis=1)
        return mean, logvar

    def reparameterize(self, mean, logvar):
        eps = tf.random.normal(shape=mean.shape)
        return eps * tf.exp(logvar * .5) + mean

    def decode(self, z, apply_sigmoid=False):
        logits = self.decoder(z)
        if apply_sigmoid:
            probs = tf.sigmoid(logits)
            return probs
        return logits

    def call(self, x):
        mean, logvar = self.encode(x)
        z = self.reparameterize(mean, logvar)
        reconstructed = self.decode(z, apply_sigmoid=True)
        return reconstructed

model = CVAE(latent_dim)

In [ ]:
# Build the model using input shapes
model.build(input_shape=(None, 256, 256, 1))

model.load_weights(model_path)

### Evaluation

In [ ]:
def get_bounding_box(ground_truth_map):
    # get bounding box from mask
    if len(ground_truth_map.shape) > 2:
        ground_truth_map = np.squeeze(ground_truth_map)
    else:
        ground_truth_map = ground_truth_map

    y_indices, x_indices = np.where(ground_truth_map > 0)

    if len(x_indices) == 0 or len(y_indices) == 0:
        return [0,0,255,255]

    x_min, x_max = np.min(x_indices), np.max(x_indices)
    y_min, y_max = np.min(y_indices), np.max(y_indices)
    # add perturbation to bounding box coordinates
    H, W = ground_truth_map.shape
    x_min = max(0, x_min - 5)
    x_max = min(W, x_max + 5)
    y_min = max(0, y_min - 5)
    y_max = min(H, y_max + 5)
    bbox = [x_min, y_min, x_max, y_max]

    return bbox

def preprocess_image(image):

    bbox = get_bounding_box(image)
    image = image[bbox[1]:bbox[3], bbox[0]:bbox[2]]
    image = cv2.resize(image, (256, 256))

    filtered_image = np.array(image)
    filtered_image = filtered_image.reshape((256, 256, 1))

    return filtered_image

def load_and_preprocess_image(file_path):
    """
    Load and preprocess a single image.
    To be used with tf.data.Dataset.map()
    """
    def read_and_process(file_path):
        # Read the image
        img = tifffile.imread(file_path.numpy().decode())
        img = img.astype(np.float32) / 4000.0
        processed_img = preprocess_image(img)

        return processed_img

    processed = tf.py_function(
        read_and_process,
        [file_path],
        tf.float32
    )

    processed.set_shape((256, 256, 1))

    return processed

def create_test_dataset(image_files, valid_ids, image_folder, batch_size=32):
    # Filter valid files and get full paths
    valid_files = [os.path.join(image_folder, f) for f in image_files
                  if f[:7] in valid_ids]

    # Create dataset from file paths
    test_dataset = tf.data.Dataset.from_tensor_slices(valid_files)

    # Set up parallel processing
    AUTOTUNE = tf.data.AUTOTUNE

    # Configure dataset
    test_dataset = test_dataset.cache()
    test_dataset = test_dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=AUTOTUNE
    )
    test_dataset = test_dataset.batch(batch_size)
    test_dataset = test_dataset.prefetch(AUTOTUNE)

    return test_dataset

def dice_coefficient(y_true, y_pred, smooth=1e-6, cutoff = 0.08):
    """Calculate Dice coefficient between two non-binary images"""
    # Binarize the images
    y_true_bin = (y_true > cutoff).astype(np.float32)
    y_pred_bin = (y_pred > cutoff).astype(np.float32)

    intersection = np.sum(y_true_bin * y_pred_bin)
    return (2. * intersection + smooth) / (np.sum(y_true_bin) + np.sum(y_pred_bin) + smooth)

def calculate_metrics(original, reconstruction):
    """Calculate all required metrics between original and reconstruction"""
    # MSE
    mse = np.mean((original - reconstruction) ** 2)

    # PSNR (using skimage)
    psnr_value = psnr(original.squeeze(), reconstruction.squeeze(), data_range=1.0)

    # SSIM (using skimage)
    ssim_value = ssim(original.squeeze(), reconstruction.squeeze(), data_range=1.0)

    # Dice coefficient
    dice = dice_coefficient(original, reconstruction)

    return {
        'mse': mse,
        'psnr': psnr_value,
        'ssim': ssim_value,
        'dice': dice
    }

def visualize_results(originals, reconstructions, indices, output_path, metrics=None):
    """
    Create side-by-side visualizations for selected image indices
    """
    n = len(indices)
    fig, axes = plt.subplots(n, 2, figsize=(10, n*5))

    for i, idx in enumerate(indices):
        # Handle the case of only one sample
        if n == 1:
            ax1, ax2 = axes[0], axes[1]
        else:
            ax1, ax2 = axes[i, 0], axes[i, 1]

        # Original
        ax1.imshow(originals[idx].squeeze(), cmap='gray')
        ax1.set_title('Original')
        ax1.axis('off')

        # Reconstruction
        ax2.imshow(reconstructions[idx].squeeze(), cmap='gray')
        ax2.set_title('Reconstruction')
        ax2.axis('off')

        # Add metrics if provided
        if metrics:
            if n == 1:
                plt.figtext(0.5, 0.01,
                        f"MSE: {metrics[idx]['mse']:.4f}, PSNR: {metrics[idx]['psnr']:.2f} dB, "
                        f"SSIM: {metrics[idx]['ssim']:.4f}, Dice: {metrics[idx]['dice']:.4f}",
                        ha="center", fontsize=10, bbox={"facecolor":"white", "alpha":0.5})
            else:
                plt.figtext(0.5, 0.01 + i*(1.0/n),
                        f"MSE: {metrics[idx]['mse']:.4f}, PSNR: {metrics[idx]['psnr']:.2f} dB, "
                        f"SSIM: {metrics[idx]['ssim']:.4f}, Dice: {metrics[idx]['dice']:.4f}",
                        ha="center", fontsize=10, bbox={"facecolor":"white", "alpha":0.5})

    plt.tight_layout()
    plt.savefig(os.path.join(output_path, "reconstructions.png"), dpi=300)
    plt.close()

def plot_pixel_distributions(originals, reconstructions, output_path):
    """Plot histograms of pixel values for originals and reconstructions"""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Flatten all images into 1D arrays
    orig_pixels = originals.reshape(-1)
    recon_pixels = reconstructions.reshape(-1)

    # Plot histograms
    axes[0].hist(orig_pixels, bins=50, alpha=0.7, color='blue')
    axes[0].set_title('Original Image Pixel Distribution')
    axes[0].set_xlabel('Pixel Value')
    axes[0].set_ylabel('Frequency')

    axes[1].hist(recon_pixels, bins=50, alpha=0.7, color='orange')
    axes[1].set_title('Reconstructed Image Pixel Distribution')
    axes[1].set_xlabel('Pixel Value')
    axes[1].set_ylabel('Frequency')

    plt.tight_layout()
    plt.savefig(os.path.join(output_path, "pixel_distributions.png"), dpi=300)
    plt.close()

    # Also create a combined histogram
    plt.figure(figsize=(10, 6))
    plt.hist(orig_pixels, bins=50, alpha=0.5, color='blue', label='Original')
    plt.hist(recon_pixels, bins=50, alpha=0.5, color='orange', label='Reconstruction')
    plt.title('Pixel Value Distribution Comparison')
    plt.xlabel('Pixel Value')
    plt.ylabel('Frequency')
    plt.legend()
    plt.savefig(os.path.join(output_path, "combined_pixel_distribution.png"), dpi=300)
    plt.close()

def plot_pixel_scatter(originals, reconstructions, output_path):
    """Plot scatter plot of pixel values in reconstructions vs originals"""
    idx = np.random.choice(len(originals), 3, replace=False)
    orig_sample = originals[idx]
    recon_sample = reconstructions[idx]

    org_sample = np.concatenate(orig_sample, axis=0)
    recon_sample = np.concatenate(recon_sample, axis=0)

    # Flatten all images into 1D arrays
    orig_pixels = org_sample.reshape(-1)
    recon_pixels = recon_sample.reshape(-1)

    # Create scatter plot
    plt.figure(figsize=(10, 8))

    # If there are too many pixels, sample a subset to avoid overcrowding the plot
    if len(orig_pixels) > 100000:
        idx = np.random.choice(len(orig_pixels), 100000, replace=False)
        orig_sample = orig_pixels[idx]
        recon_sample = recon_pixels[idx]
    else:
        orig_sample = orig_pixels
        recon_sample = recon_pixels

    # Plot scatter
    plt.scatter(orig_sample, recon_sample, alpha=0.5, s=2)

    # Add diagonal line representing perfect reconstruction
    max_val = max(orig_pixels.max(), recon_pixels.max())
    min_val = min(orig_pixels.min(), recon_pixels.min())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')

    plt.title('Original vs Reconstructed Pixel Values')
    plt.xlabel('Original Pixel Value')
    plt.ylabel('Reconstructed Pixel Value')

    # Set equal aspect ratio
    plt.axis('equal')
    plt.grid(True, alpha=0.3)

    # Save the figure
    plt.savefig(os.path.join(output_path, "pixel_scatter_plot.png"), dpi=300)
    plt.close()

In [ ]:
import os

# Load quality control data
image_files = sorted([f for f in os.listdir(image_folder) if f.endswith('.tiff')])

#OPTIONAL IMAGE FILE CROP
image_files = image_files[:1000]

# Load quality control data
mean_T1_original = pd.read_csv(quality_control_path)
valid_ids = mean_T1_original.loc[mean_T1_original['quality_controlled'], 'id'].astype(str).tolist()

valid_image_files = [file for file in image_files if file[:7] in valid_ids]

# Create test dataset
test_dataset = create_test_dataset(
    image_files=valid_image_files,
    valid_ids=valid_ids,
    image_folder=image_folder,
    batch_size=batch_size
)

In [ ]:
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Lists to store results
all_originals = []
all_reconstructions = []
all_metrics = []

for batch in tqdm(test_dataset, desc="Processing batches"):
    # Get reconstructions
    reconstructions = model(batch)

    # Convert to numpy for metric calculation
    originals_np = batch.numpy()
    reconstructions_np = reconstructions.numpy()

    # Store images
    all_originals.append(originals_np)
    all_reconstructions.append(reconstructions_np)

    # Calculate metrics for each image in batch
    for i in range(originals_np.shape[0]):
        metrics = calculate_metrics(originals_np[i], reconstructions_np[i])
        all_metrics.append(metrics)

In [ ]:
# Concatenate all batches
all_originals = np.concatenate(all_originals, axis=0)
all_reconstructions = np.concatenate(all_reconstructions, axis=0)

# Calculate average metrics
avg_metrics = {
    'mse': np.mean([m['mse'] for m in all_metrics]),
    'psnr': np.mean([m['psnr'] for m in all_metrics]),
    'ssim': np.mean([m['ssim'] for m in all_metrics]),
    'dice': np.mean([m['dice'] for m in all_metrics])
}

# Display and save average metrics
print("\nAverage Metrics:")
print(f"MSE: {avg_metrics['mse']:.5f}")
print(f"PSNR: {avg_metrics['psnr']:.2f} dB")
print(f"SSIM: {avg_metrics['ssim']:.4f}")
print(f"Dice Score: {avg_metrics['dice']:.4f}")

In [ ]:
#print image and reconstruction

idx = np.random.choice(len(all_originals),replace=False)

idx = 19

plt.figure(figsize=(15, 15))
plt.subplot(1, 2, 1)
plt.imshow(all_originals[idx].squeeze(), cmap='gray')
plt.title('Original')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(all_reconstructions[idx].squeeze(), cmap='gray')
plt.title('Reconstruction')
plt.axis('off')
plt.show()

In [ ]:
# Save metrics to CSV
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(os.path.join(output_directory, "metrics_2.csv"), index=False)

# Save summary metrics
with open(os.path.join(output_directory, "summary_metrics_2.txt"), 'w') as f:
    f.write(f"MSE: {avg_metrics['mse']:.4f}\n")
    f.write(f"PSNR: {avg_metrics['psnr']:.2f} dB\n")
    f.write(f"SSIM: {avg_metrics['ssim']:.4f}\n")
    f.write(f"Dice Score: {avg_metrics['dice']:.4f}\n")

# Visualize sample images
print("\nGenerating visualizations...")
sample_indices = np.random.choice(len(all_originals), min(num_samples, len(all_originals)), replace=False)
visualize_results(all_originals, all_reconstructions, sample_indices, output_directory, all_metrics)

# Plot pixel distributions
plot_pixel_distributions(all_originals, all_reconstructions, output_directory)

print(f"\nEvaluation complete. Results saved to {output_directory}")

In [ ]:
plot_pixel_scatter(all_originals, all_reconstructions, output_directory)

In [ ]:
def analyze_dimension_importance_in_chunks(model, test_dataset, latent_dims):
    """Analyze dimension importance by processing the dataset in small chunks."""
    n_dims = latent_dims

    # Initialize arrays to accumulate results
    total_samples = 0
    sum_reconstruction_impact = np.zeros(n_dims)
    sum_perturbation_sensitivity = np.zeros(n_dims)
    sum_kl_divergence = np.zeros(n_dims)

    # Process dataset in chunks
    max_chunk_size = 16  # Adjust based on GPU memory

    for batch in tqdm(test_dataset, desc="Analyzing dimensions"):
        # Encode batch
        mean, logvar = model.encode(batch)
        z = model.reparameterize(mean, logvar)
        base_predictions = model.sample(z)
        base_reconstruction_error = tf.reduce_mean(tf.square(batch - base_predictions))

        batch_size = batch.shape[0]
        total_samples += batch_size

        # Analyze each dimension
        for dim in range(n_dims):
            # Test reconstruction impact of zeroing out dimension
            modified_z = z.numpy().copy()
            modified_z[:, dim] = 0
            modified_predictions = model.sample(modified_z)
            dim_impact = tf.reduce_mean(tf.square(batch - modified_predictions)) - base_reconstruction_error
            sum_reconstruction_impact[dim] += float(dim_impact) * batch_size

            # Test sensitivity to perturbations
            epsilon = 0.1
            perturbed_z = z.numpy().copy()
            perturbed_z[:, dim] += epsilon
            perturbed_predictions = model.sample(perturbed_z)
            sensitivity = tf.reduce_mean(tf.abs(perturbed_predictions - base_predictions)) / epsilon
            sum_perturbation_sensitivity[dim] += float(sensitivity) * batch_size

            # Calculate KL divergence for this dimension
            kl_div = -0.5 * tf.reduce_mean(1 + logvar[:, dim] - tf.square(mean[:, dim]) - tf.exp(logvar[:, dim]))
            sum_kl_divergence[dim] += float(kl_div) * batch_size

        # Clear memory
        tf.keras.backend.clear_session()

    # Calculate averages
    reconstruction_impact = sum_reconstruction_impact / total_samples
    perturbation_sensitivity = sum_perturbation_sensitivity / total_samples
    kl_divergence = sum_kl_divergence / total_samples

    # Normalize metrics
    reconstruction_impact = reconstruction_impact / np.max(reconstruction_impact) if np.max(reconstruction_impact) > 0 else reconstruction_impact
    perturbation_sensitivity = perturbation_sensitivity / np.max(perturbation_sensitivity) if np.max(perturbation_sensitivity) > 0 else perturbation_sensitivity
    kl_divergence = kl_divergence / np.max(kl_divergence) if np.max(kl_divergence) > 0 else kl_divergence

    return reconstruction_impact, perturbation_sensitivity, kl_divergence


In [ ]:
reconstruction_impact, perturbation_sensitivity, kl_divergence = analyze_dimension_importance_in_chunks(
    model, test_dataset, 16)

In [ ]:
results = pd.DataFrame({
    'Dimension': range(model.latent_dim),
    'Reconstruction Impact': reconstruction_impact,
    'Perturbation Sensitivity': perturbation_sensitivity,
    'KL Divergence': kl_divergence })

results

In [ ]:
import tensorflow_probability as tfp

def plot_latent_images(model, n, digit_size=256, dimensions=(0, 1)):
  """Plots n x n digit images decoded from the latent space."""

  latent_dim = model.latent_dim

  if max(dimensions) >= latent_dim:
      raise ValueError(f"Specified dimensions {dimensions} exceed latent_dim={latent_dim}.")

  norm = tfp.distributions.Normal(0, 1)
  grid_x = norm.quantile(np.linspace(0.05, 0.95, n))
  grid_y = norm.quantile(np.linspace(0.05, 0.95, n))
  image_width = digit_size*n
  image_height = image_width
  image = np.zeros((image_height, image_width))

  for i, yi in enumerate(grid_x):
    for j, xi in enumerate(grid_y):
      z = np.zeros((1, latent_dim))

      z[0, dimensions[0]] = xi  # Vary the first chosen dimension
      z[0, dimensions[1]] = yi  # Vary the second chosen dimension

      x_decoded = model.sample(z)
      digit = tf.reshape(x_decoded[0], (digit_size, digit_size))
      image[i * digit_size: (i + 1) * digit_size,
            j * digit_size: (j + 1) * digit_size] = digit.numpy()

  plt.figure(figsize=(10, 10))
  plt.imshow(image, cmap='coolwarm')
  plt.axis('Off')
  plt.show()

In [ ]:
plot_latent_images(model, n=6, digit_size=256, dimensions=(1, 5))

In [ ]:
plot_latent_images(model, n=6, digit_size=256, dimensions=(10, 13))

In [ ]:
plot_latent_images(model, n=6, digit_size=256, dimensions=(14, 13))

In [ ]:
plot_latent_images(model, n=6, digit_size=256, dimensions=(2, 13))

## Visualize Latent Spaces

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
latent_spaces = pd.read_csv("./data/results_16dVAE/VAE_16_latent_dimensions_phenotypes_imputed.no_outliers.residuals.qnorm.txt", sep = "\t")

In [ ]:
# Mean, and range of each latent dimentions

# plot distribution of a latent dimension

x = "5"
pheno = 'Latent_' + x
title = 'Distribution of Latent Dimension ' + x

plt.figure(figsize=(10, 6))
plt.hist(latent_spaces[pheno], bins=50, color='skyblue', edgecolor='black')
plt.title(title)
plt.xlabel('Value')
plt.ylabel('Frequency')

# plot of overlayed histograms

plt.figure(figsize=(10, 6))
for pheno in ['Latent_' + str(i) for i in range(16)]:
    plt.hist(latent_spaces[pheno], bins=50, alpha=0.5, label=pheno)
    plt.legend(loc='upper right')
    plt.title('Distribution of Latent Dimensions')
    plt.xlabel('Value')
    plt.ylabel('Frequency')

In [ ]:
def find_optimal_k(X, max_k=10):
    """Find the optimal number of clusters using silhouette score"""
    silhouette_scores = []
    k_values = range(2, max_k+1)

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        score = silhouette_score(X, labels)
        silhouette_scores.append(score)
        print(f"K = {k}, Silhouette Score = {score:.3f}")

    best_k = k_values[np.argmax(silhouette_scores)]
    print(f"Best number of clusters: {best_k}")
    return best_k

In [ ]:
def cluster_latent_space(df, n_clusters=None):
    """Cluster the latent space and return DataFrame with cluster assignments"""
    # Extract latent columns (excluding FID and IID)
    latent_cols = [col for col in df.columns if col.startswith('Latent_')]
    X = df[latent_cols].values

    # Find optimal k if not specified
    if n_clusters is None:
        n_clusters = find_optimal_k(X)

    # Apply KMeans clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_predict(X)

    # Visualize the first 2 dimensions
    plt.figure(figsize=(10, 8))
    plt.scatter(X[:, 0], X[:, 1], c=df['cluster'], cmap='viridis', alpha=0.8)
    plt.title('Clusters in Latent Space (First 2 Dimensions)')
    plt.xlabel('Latent_0')
    plt.ylabel('Latent_1')
    plt.colorbar(label='Cluster')
    plt.grid(alpha=0.3)
    plt.show()

    return df

In [ ]:
# Cluster in latent space
latent_spaces = cluster_latent_space(latent_spaces, n_clusters=None)

latent_spaces

## Attention Maps

In [ ]:
class SimpleVAEAttentionMaps:
    """A simpler implementation of attention map generation for VAE"""

    def __init__(self, model, latent_dim=16):
        self.model = model
        self.latent_dim = latent_dim

        # We'll track the outputs of conv layers manually
        self.conv_outputs = []

    def _get_conv_outputs(self, input_image):
        """Get outputs of all convolutional layers in the encoder"""
        self.conv_outputs = []

        # Need to register a hook for each convolutional layer
        # Since we're using TensorFlow, we'll call each layer directly
        x = input_image

        # Process through each layer and save conv outputs
        for layer in self.model.encoder.layers:
            x = layer(x)
            if isinstance(layer, tf.keras.layers.Conv2D):
                self.conv_outputs.append(x)

        return x  # Return final encoder output

    def get_attention_map(self, input_image, latent_idx):
        """Generate attention map for a specific latent dimension"""
        # Convert input_image to tensor if needed
        if not isinstance(input_image, tf.Tensor):
            input_image = tf.convert_to_tensor(input_image, dtype=tf.float32)

        # Add batch dimension if needed
        if len(input_image.shape) == 3:
            input_image = tf.expand_dims(input_image, axis=0)

        # Track gradients of the latent dimension with respect to the last conv layer
        with tf.GradientTape() as tape:
            # Get all conv layer outputs
            encoder_output = self._get_conv_outputs(input_image)

            # Get the last conv layer output
            last_conv_output = self.conv_outputs[-1]

            # Keep track of the last conv layer
            tape.watch(last_conv_output)

            # Get mean vector (first half of encoder output)
            mean, _ = tf.split(encoder_output, num_or_size_splits=2, axis=1)

            # Get the specific latent dimension
            target_output = mean[:, latent_idx]

        # Calculate gradients
        grads = tape.gradient(target_output, last_conv_output)

        # Calculate weights using global average pooling
        weights = tf.reduce_mean(grads, axis=(1, 2))

        # Create weighted feature map
        cam = tf.reduce_sum(
            tf.multiply(weights[:, tf.newaxis, tf.newaxis, :], last_conv_output),
            axis=-1
        )

        # Apply ReLU
        cam = tf.nn.relu(cam)

        # Normalize
        cam = cam / (tf.reduce_max(cam) + tf.keras.backend.epsilon())

        # Convert to numpy and remove batch dimension
        cam = cam.numpy()[0]

        # Resize to match input dimensions
        cam = cv2.resize(cam, (input_image.shape[2], input_image.shape[1]))

        return cam

    def visualize_attention_maps(self, input_image, save_path=None):
        """Visualize attention maps for all latent dimensions"""
        # Generate attention maps for all latent dimensions
        attention_maps = []

        for i in range(self.latent_dim):
            try:
                attention_map = self.get_attention_map(input_image, i)
                attention_maps.append(attention_map)
            except Exception as e:
                print(f"Error generating attention map for latent dim {i}: {e}")
                # Insert a blank map as placeholder
                attention_maps.append(np.zeros((input_image.shape[1], input_image.shape[2])))

        # Calculate grid size
        grid_size = int(np.ceil(np.sqrt(self.latent_dim + 1)))

        # Create figure
        fig, axs = plt.subplots(grid_size, grid_size, figsize=(15, 15))
        axs = axs.flatten()

        # Plot original image
        axs[0].imshow(np.squeeze(input_image), cmap='gray')
        axs[0].set_title('Original Image')
        axs[0].axis('off')

        # Plot attention maps
        for i, attention_map in enumerate(attention_maps):
            axs[i+1].imshow(attention_map, cmap='jet')
            axs[i+1].set_title(f'Latent Dim {i}')
            axs[i+1].axis('off')

        # Remove empty subplots
        for i in range(self.latent_dim + 1, grid_size * grid_size):
            if i < len(axs):
                fig.delaxes(axs[i])

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')

        plt.show()

        return attention_maps

    def visualize_single_latent(self, input_image, latent_idx, save_path=None):
        """Visualize attention map for a single latent dimension with overlay"""
        attention_map = self.get_attention_map(input_image, latent_idx)

        # Create heatmap overlay
        img = np.squeeze(input_image.copy())

        # Convert to RGB for overlay
        img_rgb = np.repeat(img[..., np.newaxis], 3, axis=2)

        # Create colored heatmap
        heatmap = cv2.applyColorMap(np.uint8(255 * attention_map), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

        # Create overlay
        overlay = (0.7 * heatmap + 0.3 * img_rgb * 255).astype(np.uint8)

        # Display
        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        plt.imshow(img, cmap='gray')
        plt.title('Original Image')
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(attention_map, cmap='jet')
        plt.title(f'Attention Map (Latent {latent_idx})')
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(overlay)
        plt.title('Overlay')
        plt.axis('off')

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')

        plt.show()

        return attention_map, overlay

In [ ]:
# Get a list of images to process
image_files = sorted([f for f in os.listdir(image_folder) if f.endswith('.tiff')])
print(f"Found {len(image_files)} images in {image_folder}")

LATENT_DIM = 16

In [ ]:
# Create attention map generator
attention_generator = SimpleVAEAttentionMaps(model, LATENT_DIM)

# Process a few sample images
num_samples = 5  # Adjust as needed
sample_indices = np.random.choice(len(image_files), min(num_samples, len(image_files)), replace=False)

for idx in sample_indices:
    try:
        # Load and preprocess image
        image_path = os.path.join(image_folder, image_files[idx])
        print(f"Processing {image_path}")

        # Load image
        img = tifffile.imread(image_path)
        img = img.astype(np.float32) / 4000.0  # Normalize
        processed_img = preprocess_image(img)

        # Generate attention maps for all latent dimensions
        try:
            attention_maps = attention_generator.visualize_attention_maps(
                processed_img,
                save_path=os.path.join(output_directory, f"{os.path.basename(image_path)}_attention_maps.png")
            )

            # Create individual visualizations for a few selected dimensions
            for latent_idx in range(min(5, LATENT_DIM)):  # Just do first 5 to save time
                try:
                    attention_map, overlay = attention_generator.visualize_single_latent(
                        processed_img,
                        latent_idx,
                        save_path=os.path.join(output_directory, f"{os.path.basename(image_path)}_latent{latent_idx}_overlay.png")
                    )
                except Exception as e:
                    print(f"Error visualizing latent dimension {latent_idx}: {e}")
        except Exception as e:
            print(f"Error generating attention maps: {e}")

        # Generate basic reconstruction to visualize what the model is doing
        try:
            # Get reconstruction
            mean, logvar = model.encode(tf.expand_dims(processed_img, 0))
            z = model.reparameterize(mean, logvar)
            reconstruction = model.decode(z, apply_sigmoid=True)

            # Display original and reconstruction
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.imshow(np.squeeze(processed_img), cmap='gray')
            plt.title('Original')
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.imshow(np.squeeze(reconstruction), cmap='gray')
            plt.title('Reconstruction')
            plt.axis('off')

            plt.tight_layout()
            plt.savefig(os.path.join(output_directory, f"{os.path.basename(image_path)}_reconstruction.png"), dpi=300)
            plt.show()
        except Exception as e:
            print(f"Error generating reconstruction: {e}")

    except Exception as e:
        print(f"Error processing {image_files[idx]}: {e}")
        continue

print("\nAttention map generation complete!")

# Try to set up an interactive widget for exploration if in a notebook
try:
    from ipywidgets import interact, IntSlider, Dropdown

    # Load a few sample images
    sample_paths = [os.path.join(image_folder, image_files[i]) for i in sample_indices]
    sample_images = []
    for path in sample_paths:
        try:
            img = tifffile.imread(path)
            img = img.astype(np.float32) / 4000.0
            processed_img = preprocess_image(img)
            sample_images.append(processed_img)
        except Exception as e:
            print(f"Error loading {path}: {e}")

    # Only proceed if we have sample images
    if sample_images:
        # Create interactive widget
        def explore_attention(image_idx=0, latent_idx=0):
            image = sample_images[image_idx]
            attention_generator.visualize_single_latent(image, latent_idx)
            print(f"Showing latent dimension {latent_idx} for image {os.path.basename(sample_paths[image_idx])}")

        interact(
            explore_attention,
            image_idx=IntSlider(min=0, max=len(sample_images)-1, step=1, value=0, description='Image:'),
            latent_idx=IntSlider(min=0, max=LATENT_DIM-1, step=1, value=0, description='Latent Dim:')
        )
    else:
        print("No sample images were loaded successfully for interactive exploration.")
except ImportError:
    print("Interactive widgets require ipywidgets to be installed.")
except Exception as e:
    print(f"Error setting up interactive widget: {e}")